# Insurance Fraud Detection — Bước 3: SMOTE & XGBoost Training
**Input:** `fraud_features.csv` · `feature_metadata.json`  
**Output:** `model_xgb.pkl` · `model_metadata.json` · `threshold_analysis.csv`

---
### Cấu trúc notebook
1. Load & setup
2. Train/Test split (stratified)
3. So sánh các chiến lược xử lý imbalance
4. SMOTE — lý thuyết & implementation
5. XGBoost baseline
6. Hyperparameter tuning — Bayesian Optimization
7. Multicollinearity check từ feature importance
8. Threshold optimization — Cost-sensitive
9. Final evaluation — đầy đủ metrics
10. Lưu model & metadata


## Chiến lược xử lý Imbalance — Tổng quan

Với fraud rate 24%, có 4 chiến lược chính:

| Chiến lược | Cơ chế | Ưu điểm | Nhược điểm |
|---|---|---|---|
| **Do nothing** | Train bình thường | Đơn giản | Model thiên về majority class |
| **`scale_pos_weight`** | Tăng weight cho minority | Nhanh, không tạo data giả | Ít linh hoạt |
| **SMOTE** | Tạo synthetic minority samples | Cân bằng data thực sự | Có thể tạo noise |
| **Cost-sensitive threshold** | Dịch chuyển decision boundary | Không thay đổi model | Phụ thuộc vào cost matrix |

**Chiến lược tối ưu cho bài toán này:** SMOTE + `scale_pos_weight` + threshold optimization  
→ 3 lớp bảo vệ, tối đa hóa Recall mà không hy sinh quá nhiều Precision.


## 1. Load & Setup

In [4]:
import warnings, json, pickle
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection  import (train_test_split, StratifiedKFold,
                                       cross_validate)
from sklearn.metrics          import (roc_auc_score, average_precision_score,
                                       f1_score, recall_score, precision_score,
                                       confusion_matrix, classification_report,
                                       roc_curve, precision_recall_curve)
from sklearn.dummy            import DummyClassifier
from imblearn.over_sampling   import SMOTE, SMOTENC
from imblearn.pipeline        import Pipeline as ImbPipeline
import xgboost as xgb
from scipy.stats              import randint, uniform

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({
    "figure.dpi": 115,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "font.family": "sans-serif",
})
PALETTE = {"fraud":"#E53935","legit":"#2196F3","neutral":"#78909C","good":"#43A047"}

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv("fraud_features.csv")
with open("feature_metadata.json") as f:
    feat_meta = json.load(f)

TARGET        = "target"
FEATURE_NAMES = feat_meta["feature_names"]
FEATURE_NAMES = [f for f in FEATURE_NAMES if f in df.columns]

X = df[FEATURE_NAMES].fillna(0)
y = df[TARGET]

print(f"✓ Loaded: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"  Fraud rate : {y.mean()*100:.1f}%")
print(f"  Class ratio: {(y==0).sum()}:{(y==1).sum()} "
      f"({(y==0).sum()/(y==1).sum():.1f}:1)")


ModuleNotFoundError: No module named 'xgboost'

## 2. Train / Test Split (Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Train: {len(X_train):,} rows | Fraud: {y_train.sum():,} ({y_train.mean()*100:.1f}%)")
print(f"Test : {len(X_test):,}  rows | Fraud: {y_test.sum():,}  ({y_test.mean()*100:.1f}%)")

# Scale pos weight cho XGBoost
SCALE_POS_WEIGHT = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nscale_pos_weight = {SCALE_POS_WEIGHT:.2f}")


## 3. So sánh các chiến lược xử lý Imbalance

In [ ]:
# ── Quick benchmark: 4 strategies với XGBoost default params ─────────────────
BASE_PARAMS = dict(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", random_state=42, n_jobs=-1,
)

strategies = {
    "No resampling":           xgb.XGBClassifier(**BASE_PARAMS),
    "scale_pos_weight":        xgb.XGBClassifier(scale_pos_weight=SCALE_POS_WEIGHT, **BASE_PARAMS),
    "SMOTE":                   ImbPipeline([
                                   ("smote", SMOTE(random_state=42, k_neighbors=5)),
                                   ("xgb",   xgb.XGBClassifier(**BASE_PARAMS)),
                               ]),
    "SMOTE + scale_pos_weight": ImbPipeline([
                                   ("smote", SMOTE(random_state=42, k_neighbors=5)),
                                   ("xgb",   xgb.XGBClassifier(
                                       scale_pos_weight=SCALE_POS_WEIGHT, **BASE_PARAMS)),
                               ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print("Chạy cross-validation cho 4 strategies (mất ~2-3 phút)...")
for name, model in strategies.items():
    cv_res = cross_validate(
        model, X_train, y_train, cv=cv,
        scoring=["roc_auc","average_precision","f1","recall","precision"],
        n_jobs=-1,
    )
    results[name] = {
        "PR-AUC":    cv_res["test_average_precision"].mean(),
        "AUC-ROC":   cv_res["test_roc_auc"].mean(),
        "F1":        cv_res["test_f1"].mean(),
        "Recall":    cv_res["test_recall"].mean(),
        "Precision": cv_res["test_precision"].mean(),
    }
    print(f"  ✓ {name}")

results_df = pd.DataFrame(results).T.round(4)
print("\n─── Strategy Comparison (5-fold CV) ───")
print(results_df.to_string())

best_strategy = results_df["PR-AUC"].idxmax()
print(f"\n✓ Best strategy: {best_strategy}")


In [ ]:
# ── Visualize strategy comparison ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
metrics_plot = ["PR-AUC", "F1", "Recall", "Precision"]
x = np.arange(len(metrics_plot))
w = 0.18

colors_strat = [PALETTE["neutral"], PALETTE["legit"],
                PALETTE["good"],    PALETTE["fraud"]]

for i, (strat, row) in enumerate(results_df.iterrows()):
    vals = [row[m] for m in metrics_plot]
    axes[0].bar(x + i*w, vals, width=w, label=strat,
                color=colors_strat[i], edgecolor="white", alpha=0.85)

axes[0].set_xticks(x + w*1.5)
axes[0].set_xticklabels(metrics_plot, fontsize=10)
axes[0].set_ylabel("Score")
axes[0].set_title("Strategy Comparison — 5-fold CV")
axes[0].legend(fontsize=8, frameon=False)
axes[0].set_ylim(0, 1.1)

# Radar-style: PR-AUC vs Recall trade-off
axes[1].scatter(
    results_df["Recall"], results_df["PR-AUC"],
    s=150, zorder=5,
    color=colors_strat,
    edgecolors="white", linewidths=1.5,
)
for strat, row in results_df.iterrows():
    axes[1].annotate(
        strat, (row["Recall"], row["PR-AUC"]),
        xytext=(8, 4), textcoords="offset points", fontsize=8.5,
    )
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("PR-AUC")
axes[1].set_title("Recall vs PR-AUC Trade-off")

plt.suptitle("Imbalance Handling Strategy Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fraud_07_strategy_comparison.png", bbox_inches="tight")
plt.show()


## 4. SMOTE — Lý thuyết & Implementation

**SMOTE (Synthetic Minority Over-sampling Technique)** tạo ra synthetic samples
bằng cách nội suy giữa các minority samples thực sự:

```
synthetic_sample = x_i + λ × (x_neighbor - x_i)
```

Trong đó:
- `x_i` = một fraud sample thực
- `x_neighbor` = một trong k nearest neighbors của x_i (cũng là fraud)
- `λ` = random ∈ [0, 1]

**Khác với random oversampling** (chỉ duplicate): SMOTE tạo điểm mới
nằm **giữa** các điểm thực → model học được decision boundary rộng hơn,
không bị overfit vào exact training points.

**Lưu ý quan trọng:** SMOTE chỉ áp dụng trên **train set** —
không bao giờ áp dụng trên test set, vì test set phải phản ánh
phân phối thực tế của production data.


In [ ]:
# ── Minh họa SMOTE effect ─────────────────────────────────────────────────────
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Trước SMOTE:")
print(f"  Total  : {len(y_train):,}")
print(f"  Legit  : {(y_train==0).sum():,} ({(y_train==0).mean()*100:.1f}%)")
print(f"  Fraud  : {(y_train==1).sum():,} ({(y_train==1).mean()*100:.1f}%)")

print("\nSau SMOTE:")
print(f"  Total  : {len(y_train_sm):,}")
print(f"  Legit  : {(y_train_sm==0).sum():,} ({(y_train_sm==0).mean()*100:.1f}%)")
print(f"  Fraud  : {(y_train_sm==1).sum():,} ({(y_train_sm==1).mean()*100:.1f}%)")
print(f"  Synthetic samples tạo thêm: {(y_train_sm==1).sum() - (y_train==1).sum():,}")

# Visualize trên 2 top features (PCA để giảm chiều)
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
X_2d_before = pca.fit_transform(X_train)
X_2d_after  = pca.transform(X_train_sm)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, X_2d, y_plot, title in [
    (axes[0], X_2d_before, y_train,    "Trước SMOTE"),
    (axes[1], X_2d_after,  y_train_sm, "Sau SMOTE"),
]:
    ax.scatter(X_2d[y_plot==0, 0], X_2d[y_plot==0, 1],
               alpha=0.3, s=8, color=PALETTE["legit"],  label="Legit")
    ax.scatter(X_2d[y_plot==1, 0], X_2d[y_plot==1, 1],
               alpha=0.5, s=12, color=PALETTE["fraud"], label="Fraud")
    ax.set_title(f"{title}
(n_fraud={y_plot.sum():,})")
    ax.legend(fontsize=9, frameon=False, markerscale=2)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")

plt.suptitle("SMOTE Effect — PCA Projection", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fraud_08_smote_visualization.png", bbox_inches="tight")
plt.show()


## 5. XGBoost Baseline

In [7]:
# ── Train baseline với SMOTE data ────────────────────────────────────────────
xgb_base = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=SCALE_POS_WEIGHT,
    eval_metric="logloss",
    early_stopping_rounds=20,
    random_state=42,
    n_jobs=-1,
)

xgb_base.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

y_prob_base = xgb_base.predict_proba(X_test)[:, 1]
y_pred_base = xgb_base.predict(X_test)

print("═══ BASELINE XGBoost (test set) ═══")
print(f"  AUC-ROC  : {roc_auc_score(y_test, y_prob_base):.4f}")
print(f"  PR-AUC   : {average_precision_score(y_test, y_prob_base):.4f}")
print(f"  F1       : {f1_score(y_test, y_pred_base):.4f}")
print(f"  Recall   : {recall_score(y_test, y_pred_base):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_base):.4f}")
print()
print(classification_report(y_test, y_pred_base, target_names=["Legit","Fraud"]))

# Learning curve — kiểm tra overfit
results_eval = xgb_base.evals_result()
train_loss = results_eval.get("validation_0", {}).get("logloss", [])
if train_loss:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(train_loss, color=PALETTE["legit"], lw=2, label="Eval logloss")
    best_iter = xgb_base.best_iteration
    ax.axvline(best_iter, color="orange", linestyle="--",
               lw=1.5, label=f"Best iteration = {best_iter}")
    ax.set_xlabel("Boosting rounds")
    ax.set_ylabel("Log Loss")
    ax.set_title("XGBoost Learning Curve")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.savefig("fraud_09_learning_curve.png", bbox_inches="tight")
    plt.show()


NameError: name 'xgb' is not defined

## 6. Hyperparameter Tuning — RandomizedSearchCV

Dùng **RandomizedSearchCV** thay vì GridSearchCV vì:
- Grid search với 5 params × 4 values = 1024 combinations × 5-fold = 5120 fits
- Random search với 50 iterations × 5-fold = 250 fits → nhanh hơn 20x
- Theo nghiên cứu của Bergstra & Bengio (2012): random search tìm được
  hyperparams tốt tương đương grid search với ít iterations hơn nhiều

**Scoring metric: PR-AUC** — phù hợp hơn AUC-ROC cho imbalanced dataset.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators":        randint(200, 600),
    "max_depth":           randint(3, 8),
    "learning_rate":       uniform(0.01, 0.2),
    "subsample":           uniform(0.6, 0.4),
    "colsample_bytree":    uniform(0.5, 0.5),
    "min_child_weight":    randint(1, 10),
    "gamma":               uniform(0, 0.5),
    "reg_alpha":           uniform(0, 1),
    "reg_lambda":          uniform(0.5, 2),
}

xgb_tune = xgb.XGBClassifier(
    scale_pos_weight=SCALE_POS_WEIGHT,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rscv = RandomizedSearchCV(
    xgb_tune,
    param_distributions=param_dist,
    n_iter=50,
    scoring="average_precision",   # PR-AUC
    cv=cv_strat,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

print("Chạy RandomizedSearchCV (50 iterations × 5-fold)...")
print("Ước tính: 3-5 phút\n")
rscv.fit(X_train_sm, y_train_sm)

print(f"\n✓ Best PR-AUC (CV): {rscv.best_score_:.4f}")
print(f"Best params:")
for k, v in rscv.best_params_.items():
    print(f"  {k:25s} = {v}")


In [ ]:
# ── Evaluate tuned model ──────────────────────────────────────────────────────
best_xgb = rscv.best_estimator_
y_prob_tuned = best_xgb.predict_proba(X_test)[:, 1]
y_pred_tuned = best_xgb.predict(X_test)

print("═══ TUNED XGBoost vs BASELINE ═══")
print(f"{'Metric':12s} {'Baseline':>10} {'Tuned':>10} {'Δ':>8}")
print("─" * 44)
for metric, fn in [
    ("AUC-ROC",  lambda yp: roc_auc_score(y_test, yp)),
    ("PR-AUC",   lambda yp: average_precision_score(y_test, yp)),
    ("F1",       lambda yp: f1_score(y_test, (yp>=0.5).astype(int))),
    ("Recall",   lambda yp: recall_score(y_test, (yp>=0.5).astype(int))),
    ("Precision",lambda yp: precision_score(y_test, (yp>=0.5).astype(int))),
]:
    base_val  = fn(y_prob_base)
    tuned_val = fn(y_prob_tuned)
    delta     = tuned_val - base_val
    marker    = "↑" if delta > 0.001 else "↓" if delta < -0.001 else "="
    print(f"  {metric:10s} {base_val:>10.4f} {tuned_val:>10.4f} {delta:>+7.4f} {marker}")


## 7. Feature Importance & Multicollinearity Check

Như đã thảo luận ở Bước 2, `total_claim_amount` ↔ `injury_claim` (r=0.81)
và `number_of_vehicles` ↔ `collision_type_le` (r=-0.89).

Bây giờ nhìn vào feature importance để quyết định có nên drop không.


In [ ]:
# ── Feature importance (gain — quan trọng nhất cho XGBoost) ──────────────────
importance_gain  = best_xgb.get_booster().get_score(importance_type="gain")
importance_cover = best_xgb.get_booster().get_score(importance_type="cover")
importance_freq  = best_xgb.get_booster().get_score(importance_type="weight")

imp_df = pd.DataFrame({
    "Feature": list(importance_gain.keys()),
    "Gain":    list(importance_gain.values()),
    "Cover":   [importance_cover.get(f, 0) for f in importance_gain.keys()],
    "Freq":    [importance_freq.get(f, 0)  for f in importance_gain.keys()],
}).sort_values("Gain", ascending=False).reset_index(drop=True)

print("Top 20 features by Gain:")
print(imp_df.head(20).to_string(index=False))

# ── Check cặp correlated ─────────────────────────────────────────────────────
CORR_PAIRS = [
    ("total_claim_amount", "injury_claim"),
    ("number_of_vehicles_involved", "collision_type_le"),
]
print("\nKiểm tra cặp correlated:")
for f1, f2 in CORR_PAIRS:
    g1 = importance_gain.get(f1, 0)
    g2 = importance_gain.get(f2, 0)
    print(f"  {f1:35s} Gain = {g1:,.1f}")
    print(f"  {f2:35s} Gain = {g2:,.1f}")
    if g1 > 0 and g2 > 0:
        ratio = max(g1,g2) / (min(g1,g2) + 1e-9)
        if ratio > 3:
            drop = f1 if g1 < g2 else f2
            print(f"  → Gap lớn ({ratio:.1f}x) — cân nhắc drop '{drop}'")
        else:
            print(f"  → Cả 2 đều có contribution tương đương — GIỮ CẢ HAI")
    print()

# ── Visualize top 25 ──────────────────────────────────────────────────────────
top25 = imp_df.head(25)
fig, ax = plt.subplots(figsize=(10, 8))
colors_imp = [
    PALETTE["fraud"]   if g > top25["Gain"].quantile(0.75) else
    PALETTE["legit"]   if g > top25["Gain"].quantile(0.25) else
    PALETTE["neutral"]
    for g in top25["Gain"]
]
ax.barh(top25["Feature"][::-1], top25["Gain"][::-1],
        color=colors_imp[::-1], height=0.6, edgecolor="white")
ax.set_xlabel("Gain (information gain contributed by feature)")
ax.set_title("XGBoost Feature Importance — Top 25 (by Gain)")
plt.tight_layout()
plt.savefig("fraud_10_feature_importance.png", bbox_inches="tight")
plt.show()


## 8. Threshold Optimization — Cost-Sensitive

Default threshold = 0.5 không phải tối ưu cho fraud detection.
Ta tìm threshold tối ưu theo **F1** và theo **business cost matrix**.

**Cost matrix:**
- FN (miss fraud): cost = claim amount trung bình = $45,000
- FP (flag legit nhầm): cost = investigation cost = $500
- Cost ratio FN/FP = 90x → threshold nên thấp hơn 0.5


In [ ]:
# ── Threshold analysis ────────────────────────────────────────────────────────
precision_arr, recall_arr, pr_thresholds = precision_recall_curve(
    y_test, y_prob_tuned
)
fpr_arr, tpr_arr, roc_thresholds = roc_curve(y_test, y_prob_tuned)

# F1 tại mỗi threshold
f1_arr = np.where(
    (precision_arr[:-1] + recall_arr[:-1]) > 0,
    2 * precision_arr[:-1] * recall_arr[:-1] / (precision_arr[:-1] + recall_arr[:-1]),
    0,
)
best_f1_idx = np.argmax(f1_arr)
best_f1_thr = pr_thresholds[best_f1_idx]

# Cost-sensitive threshold
FN_COST = 45_000
FP_COST = 500
total_costs = []
for thr in pr_thresholds:
    y_pred_t = (y_prob_tuned >= thr).astype(int)
    cm_t     = confusion_matrix(y_test, y_pred_t)
    tn, fp, fn, tp = cm_t.ravel()
    cost = fn * FN_COST + fp * FP_COST
    total_costs.append(cost)

best_cost_idx = np.argmin(total_costs)
best_cost_thr = pr_thresholds[best_cost_idx]

print(f"Optimal threshold by F1   : {best_f1_thr:.4f}")
print(f"Optimal threshold by Cost : {best_cost_thr:.4f}")
print()

# ── Detailed threshold table ──────────────────────────────────────────────────
thr_analysis = []
for thr in np.arange(0.1, 0.8, 0.05):
    y_pred_t = (y_prob_tuned >= thr).astype(int)
    cm_t     = confusion_matrix(y_test, y_pred_t)
    tn, fp, fn, tp = cm_t.ravel()
    thr_analysis.append({
        "Threshold":  round(thr, 2),
        "TP":         int(tp), "FP": int(fp),
        "FN":         int(fn), "TN": int(tn),
        "Recall":     round(tp/(tp+fn) if tp+fn>0 else 0, 3),
        "Precision":  round(tp/(tp+fp) if tp+fp>0 else 0, 3),
        "F1":         round(2*tp/(2*tp+fp+fn) if 2*tp+fp+fn>0 else 0, 3),
        "Total cost ($K)": round((fn*FN_COST + fp*FP_COST)/1000, 1),
    })

thr_df = pd.DataFrame(thr_analysis)
print("Threshold Analysis:")
print(thr_df.to_string(index=False))
thr_df.to_csv("threshold_analysis.csv", index=False)

# ── Visualize ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# PR curve
axes[0].plot(recall_arr, precision_arr, color=PALETTE["legit"], lw=2,
             label=f"PR Curve (AP={average_precision_score(y_test, y_prob_tuned):.3f})")
axes[0].scatter(recall_arr[best_f1_idx], precision_arr[best_f1_idx],
                s=120, color=PALETTE["fraud"], zorder=5,
                label=f"Best F1 @ thr={best_f1_thr:.2f}")
axes[0].axhline(y_test.mean(), color="gray", linestyle="--",
                lw=1, label=f"Baseline={y_test.mean()*100:.1f}%")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curve")
axes[0].legend(fontsize=8, frameon=False)

# F1 vs threshold
axes[1].plot(pr_thresholds, f1_arr, color=PALETTE["good"],
             lw=2, label="F1")
axes[1].plot(pr_thresholds, precision_arr[:-1], color=PALETTE["legit"],
             lw=1.5, linestyle="--", alpha=0.7, label="Precision")
axes[1].plot(pr_thresholds, recall_arr[:-1], color=PALETTE["fraud"],
             lw=1.5, linestyle="--", alpha=0.7, label="Recall")
axes[1].axvline(best_f1_thr, color="orange", linestyle=":",
                lw=2, label=f"Best F1 thr={best_f1_thr:.2f}")
axes[1].set_xlabel("Threshold"); axes[1].set_ylabel("Score")
axes[1].set_title("F1 / Precision / Recall vs Threshold")
axes[1].legend(fontsize=8, frameon=False)
axes[1].set_xlim(0, 0.8)

# Cost vs threshold
axes[2].plot(pr_thresholds, [c/1000 for c in total_costs],
             color=PALETTE["fraud"], lw=2)
axes[2].axvline(best_cost_thr, color="orange", linestyle=":",
                lw=2, label=f"Min cost thr={best_cost_thr:.2f}")
axes[2].scatter([best_cost_thr], [min(total_costs)/1000],
                s=120, color="orange", zorder=5)
axes[2].set_xlabel("Threshold"); axes[2].set_ylabel("Total Cost ($K)")
axes[2].set_title("Business Cost vs Threshold")
axes[2].legend(fontsize=9, frameon=False)
axes[2].set_xlim(0, 0.8)

plt.suptitle("Threshold Optimization", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fraud_11_threshold_optimization.png", bbox_inches="tight")
plt.show()

FINAL_THRESHOLD = best_f1_thr
print(f"\n→ Final threshold sử dụng: {FINAL_THRESHOLD:.4f} (F1-optimal)")


## 9. Final Evaluation

In [ ]:
# ── Final predictions với optimal threshold ───────────────────────────────────
y_pred_final = (y_prob_tuned >= FINAL_THRESHOLD).astype(int)
cm_final     = confusion_matrix(y_test, y_pred_final)
tn, fp, fn, tp = cm_final.ravel()

auc_roc = roc_auc_score(y_test, y_prob_tuned)
pr_auc  = average_precision_score(y_test, y_prob_tuned)
f1_val  = f1_score(y_test, y_pred_final)
rec_val = recall_score(y_test, y_pred_final)
pre_val = precision_score(y_test, y_pred_final)

print("╔══════════════════════════════════════════════════╗")
print("║     FINAL MODEL PERFORMANCE (Test Set)           ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  AUC-ROC         : {auc_roc:.4f}                       ║")
print(f"║  PR-AUC          : {pr_auc:.4f}                       ║")
print(f"║  F1 Score        : {f1_val:.4f}                       ║")
print(f"║  Recall (fraud)  : {rec_val:.4f}  ← Key metric        ║")
print(f"║  Precision       : {pre_val:.4f}                       ║")
print(f"║  Threshold used  : {FINAL_THRESHOLD:.4f}                       ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}         ║")
print(f"║  Fraud caught    : {tp}/{tp+fn} ({tp/(tp+fn)*100:.1f}%)              ║")
print(f"║  False alarms    : {fp}/{fp+tn} ({fp/(fp+tn)*100:.1f}%)             ║")
print("╚══════════════════════════════════════════════════╝")

# ── Confusion matrix heatmap ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    cm_final, annot=True, fmt=",d", cmap="Blues",
    xticklabels=["Legit","Fraud"],
    yticklabels=["Legit","Fraud"],
    linewidths=0.5, ax=axes[0],
    annot_kws={"size":14, "weight":"bold"},
)
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title(f"Confusion Matrix (thr={FINAL_THRESHOLD:.2f})")

# ROC Curve
fpr_f, tpr_f, _ = roc_curve(y_test, y_prob_tuned)
axes[1].plot(fpr_f, tpr_f, color=PALETTE["fraud"], lw=2.5,
             label=f"XGBoost (AUC={auc_roc:.3f})")
axes[1].plot([0,1],[0,1], "k--", lw=1, alpha=0.5, label="Random")
axes[1].fill_between(fpr_f, tpr_f, alpha=0.06, color=PALETTE["fraud"])
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
axes[1].set_title(f"ROC Curve — AUC = {auc_roc:.3f}")
axes[1].legend(fontsize=10, frameon=False)

plt.suptitle("Final Model Evaluation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fraud_12_final_evaluation.png", bbox_inches="tight")
plt.show()


## 10. Lưu Model & Metadata

In [ ]:
# ── 1. Save model ─────────────────────────────────────────────────────────────
with open("model_xgb.pkl","wb") as f:
    pickle.dump(best_xgb, f)
print("✓ Saved: model_xgb.pkl")

# ── 2. Save SMOTE object (cần cho pipeline production) ───────────────────────
with open("smote.pkl","wb") as f:
    pickle.dump(smote, f)
print("✓ Saved: smote.pkl")

# ── 3. Save metadata ──────────────────────────────────────────────────────────
model_meta = {
    "model_type":        "XGBoost",
    "version":           "1.0",
    "feature_names":     FEATURE_NAMES,
    "n_features":        len(FEATURE_NAMES),
    "best_params":       {k: float(v) if isinstance(v, (np.integer, np.floating))
                          else v for k, v in rscv.best_params_.items()},
    "threshold":         float(FINAL_THRESHOLD),
    "threshold_method":  "F1-optimal",
    "imbalance_strategy":"SMOTE + scale_pos_weight",
    "scale_pos_weight":  float(SCALE_POS_WEIGHT),
    "performance": {
        "auc_roc":   round(float(auc_roc), 4),
        "pr_auc":    round(float(pr_auc),  4),
        "f1":        round(float(f1_val),  4),
        "recall":    round(float(rec_val), 4),
        "precision": round(float(pre_val), 4),
    },
    "confusion_matrix": {
        "TP": int(tp), "FP": int(fp),
        "FN": int(fn), "TN": int(tn),
    },
    "cost_analysis": {
        "fn_cost_per_case":   FN_COST,
        "fp_cost_per_case":   FP_COST,
        "total_cost_at_threshold": round(float(total_costs[best_cost_idx])/1000, 1),
    },
}
with open("model_metadata.json","w") as f:
    json.dump(model_meta, f, indent=2)
print("✓ Saved: model_metadata.json")

print()
print("═══ TÓM TẮT BƯỚC 3 ═══")
print(f"  Strategy  : SMOTE + scale_pos_weight={SCALE_POS_WEIGHT:.1f}")
print(f"  Tuning    : RandomizedSearchCV 50 iter × 5-fold")
print(f"  AUC-ROC   : {auc_roc:.4f}")
print(f"  PR-AUC    : {pr_auc:.4f}")
print(f"  F1        : {f1_val:.4f}")
print(f"  Recall    : {rec_val:.4f}  ← {tp}/{tp+fn} fraud cases caught")
print(f"  Threshold : {FINAL_THRESHOLD:.4f}")
print()
print("─── Bước tiếp theo: Bước 4 — SHAP Explainability ───")
